In [ ]:
import base64, os, pathlib, subprocess
from google.colab import userdata

REPO_URL = "https://github.com/devlucascfarias/logos-3.git"
REPO_BRANCH = "main"
WORKDIR = "/content/logos-3"
repo = pathlib.Path(WORKDIR)

token = userdata.get("GH_TOKEN")
git = ["git"]
if token:
    auth = base64.b64encode(f"x-access-token:{token}".encode()).decode()
    git += ["-c", f"http.extraHeader=Authorization: Basic {auth}"]

if (repo / ".git").exists():
    subprocess.run(git + ["-C", WORKDIR, "pull", "--ff-only", "origin", REPO_BRANCH], check=True)
elif repo.exists():
    raise RuntimeError(f"{WORKDIR} existe, mas não é um repositório Git")
else:
    subprocess.run(git + ["clone", "--branch", REPO_BRANCH, "--depth", "1", REPO_URL, WORKDIR], check=True)

token = auth = None
os.chdir(WORKDIR)
print(f"Repositório sincronizado em {os.getcwd()}")

# Qwen3-8B QLoRA em NVIDIA L4

Pipeline da receita SFT. Comece com `SMOKE_TEST=True`; o treino completo só deve ser iniciado depois que dados, testes e ambiente passarem.

In [ ]:
SMOKE_TEST = True
STAGE = "baseline"  # baseline, main ou agentic
RUN_TRAINING = True

TOKEN_BUDGET = 250_000 if SMOKE_TEST else None
MAX_SOURCE_ROWS = 1_000 if SMOKE_TEST else None
MAX_STEPS = 30 if SMOKE_TEST else None
MAX_TRAIN_SAMPLES = 250 if SMOKE_TEST else None
print({"stage": STAGE, "smoke": SMOKE_TEST, "token_budget": TOKEN_BUDGET})

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"], check=True)

In [ ]:
import os
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get("HF_TOKEN")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    login(token=hf_token, add_to_git_credential=False)
else:
    print("HF_TOKEN não definido; apenas fontes públicas sem aceite funcionarão.")
hf_token = None

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/environment_check.py"], check=True)

In [ ]:
command = [sys.executable, "scripts/prepare_data.py", "--stage", STAGE]
if TOKEN_BUDGET is not None:
    command += ["--token-budget", str(TOKEN_BUDGET)]
if MAX_SOURCE_ROWS is not None:
    command += ["--max-source-rows", str(MAX_SOURCE_ROWS)]
subprocess.run(command, check=True)

In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q"], check=True)

In [ ]:
if RUN_TRAINING:
    command = [sys.executable, "-u", "scripts/train_sft.py", "--stage", STAGE, "--resume-from-checkpoint", "auto"]
    if MAX_STEPS is not None:
        command += ["--max-steps", str(MAX_STEPS)]
    if MAX_TRAIN_SAMPLES is not None:
        command += ["--max-train-samples", str(MAX_TRAIN_SAMPLES)]
    print("Iniciando treino com barra de progresso e ETA...", flush=True)
    subprocess.run(command, check=True, cwd=WORKDIR)
else:
    print("RUN_TRAINING=False: dados e testes prontos; treino não iniciado.")

## Próximo passo

Avalie o modelo-base e os checkpoints no mesmo conjunto reservado. Preencha um JSON por checkpoint conforme `examples/checkpoint_metrics.example.json` e execute `scripts/select_checkpoint.py`.